
# PSD-Governed Stochastic SFH Prior: Burstiness Corner Cases

The field SFH employs a damped random walk (DRW) power spectral density (PSD)
to govern stochastic star formation history realizations. Two parameters control
the prior distribution of SFR time-variability:

- **psd_sigma** (dimensionless): amplitude of fluctuations around the mean SFH.
- **psd_tau_myr** (Myr): timescale of correlation decay.

This visualization samples 30 independent SFR realizations from the prior at
four corner cases of (psd_sigma, psd_tau_myr) and overlays percentile bands
(5th, 25th, 50th, 75th, 95th) to show how the prior responds to parameter choices.

The DRW power spectrum is:

\begin{align}P(\omega) = \sigma^2 \tau / (1 + (\tau \omega)^2)\end{align}

which produces smooth, slowly-varying SFHs for large tau (long memory) and
noisy, fast-varying SFHs for small tau (short memory). The amplitude sigma
controls the variance independent of timescale.

References:
- Iyer & Gawiser (2017), ApJ 838, 127 — Dense basis SFH reconstruction
- NIFTy correlated field formalism (Selig et al. 2013)


In [ ]:
import os
import warnings

import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# --- Grid setup ---
n_grid = 256
log_age_grid = tengri.make_log_age_grid(n_grid)
d_log_age = float(log_age_grid[1] - log_age_grid[0])
age_lookback_yr = 10.0**log_age_grid
age_lookback_gyr = np.array(age_lookback_yr) / 1e9

# --- Smooth mean SFH (shared across all panels) ---
# Use a truncated skewed normal SFH peaked at 2 Gyr lookback
mean_sfr = tengri.tsnorm(
    age_lookback_yr, log_peak_sfr=0.5, peak_lbt=2e9, width=1.5e9, skew=0.5, trunc=3.0
)
sfr_mean = np.array(mean_sfr)

# --- Corner case configurations ---
configs = [
    {
        "psd_sigma": 0.3,
        "psd_tau_myr": 50,
        "label": "Smooth, fast-varying\n" + r"($\sigma=0.3$, $\tau=50$ Myr)",
    },
    {
        "psd_sigma": 1.0,
        "psd_tau_myr": 50,
        "label": "Noisy, fast-varying\n" + r"($\sigma=1.0$, $\tau=50$ Myr)",
    },
    {
        "psd_sigma": 0.3,
        "psd_tau_myr": 300,
        "label": "Smooth, slowly-varying\n" + r"($\sigma=0.3$, $\tau=300$ Myr)",
    },
    {
        "psd_sigma": 1.0,
        "psd_tau_myr": 300,
        "label": "Bursty, slowly-varying\n" + r"($\sigma=1.0$, $\tau=300$ Myr)",
    },
]

# --- 2x2 grid of prior samples ---
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, cfg in zip(axes, configs):
    psd_sigma = cfg["psd_sigma"]
    psd_tau_myr = cfg["psd_tau_myr"]
    label = cfg["label"]

    # Compute the PSD and its square-root amplitude operator
    sqrt_power = tengri.compute_sqrt_power_drw(n_grid, d_log_age, psd_sigma, psd_tau_myr * 1e6)

    # Sample 30 realizations from the field SFH prior
    n_samples = 30
    sfr_samples = []
    key = jr.PRNGKey(42)

    for i in range(n_samples):
        key = jr.fold_in(key, i)
        # Draw a latent GP field: xi ~ N(0, I)
        xi = jnp.array(jr.normal(key, shape=(n_grid,)))

        # Convert latent to correlated GP: s = IFFT(sqrt(P) * xi)
        gp = tengri.gp_from_xi(xi, sqrt_power, n_grid)

        # Full SFH = mean * exp(GP - variance/2) for lognormal correction
        variance = float(jnp.var(gp))
        sfr = sfr_mean * jnp.exp(gp - variance / 2.0)

        sfr_samples.append(np.array(sfr))

    sfr_samples = np.array(sfr_samples)  # (n_samples, n_grid)

    # Plot individual draws as thin lines
    for sfr in sfr_samples:
        ax.plot(age_lookback_gyr, np.clip(sfr, 1e-2, 1e2), lw=0.5, alpha=0.3, color="C0")

    # Compute and plot percentile bands
    percentiles = [5, 25, 50, 75, 95]
    sfr_percs = {p: np.percentile(sfr_samples, p, axis=0) for p in percentiles}

    # 90% band (5th–95th)
    ax.fill_between(
        age_lookback_gyr,
        np.clip(sfr_percs[5], 1e-2, 1e2),
        np.clip(sfr_percs[95], 1e-2, 1e2),
        alpha=0.15,
        color="blue",
    )

    # 50% band (25th–75th)
    ax.fill_between(
        age_lookback_gyr,
        np.clip(sfr_percs[25], 1e-2, 1e2),
        np.clip(sfr_percs[75], 1e-2, 1e2),
        alpha=0.3,
        color="blue",
    )

    # Median line
    ax.plot(
        age_lookback_gyr,
        np.clip(sfr_percs[50], 1e-2, 1e2),
        color="darkblue",
        linewidth=1.5,
        label="Median (50th percentile)",
    )

    # Mean SFH overlay (dashed)
    ax.plot(age_lookback_gyr, sfr_mean, "k--", lw=1.5, label="Mean SFH", zorder=5)

    # Annotation with scenario label
    ax.text(
        0.98,
        0.97,
        label,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    )

    ax.set_xlabel("Lookback time [Gyr]", fontsize=10)
    ax.set_ylabel(r"SFR [M$_\odot$ yr$^{-1}$]", fontsize=10)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(0.01, 14)
    ax.set_ylim(1e-2, 1e2)
    ax.grid(True, alpha=0.3, which="both")

# Add shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=2, fontsize=10)

fig.tight_layout(rect=[0, 0, 1, 0.96])

script_dir = os.path.dirname(os.path.abspath(__file__))
png_path = os.path.join(script_dir, "plot_psd_burstiness_prior.png")
plt.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close()

print(f"Saved to {png_path}")